# BINARY EXPERIMENT: INCREMENTAL VS FULL RETRAIN 500

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List
import re

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# === CONFIG ===

LABEL_COL = "Is_Suspicious"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "4_class_balance").exists():
    parent = PROJECT_ROOT.parent
    if (parent / "4_class_balance").exists():
        PROJECT_ROOT = parent
    else:
        raise RuntimeError(f"Cannot find '4_class_balance' folder from cwd={Path.cwd()}")

VARIANT_ROOTS: Dict[str, Path] = {
    "baseline": PROJECT_ROOT / "4_class_balance" / "baseline_500",
    "rose":     PROJECT_ROOT / "4_class_balance" / "rose_500",
}

VARIANT_TRAIN_FILE = {
    "baseline": "train_baseline.csv",
    "rose":     "train_rose.csv",
}

# Base binary models
ALGOS = ["GBM", "XGB", "RF"]

TARGET_RECALL = 0.98
RANDOM_STATE = 42

# Optional wave filter
WAVE_FILTER = None

# === HELPERS ===
def list_waves(root: Path) -> List[str]:
    """Return sorted list of wave directory names under root."""
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])

def _resolve_train_path(dir_path: Path, variant: str) -> Path:
    """Return train file path for variant inside dir_path."""
    primary = dir_path / VARIANT_TRAIN_FILE[variant]
    if primary.exists():
        return primary
    fallback = dir_path / "train.csv"
    if fallback.exists():
        return fallback
    return primary

def load_frames(variant: str, wave: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load train/test DataFrames for given variant and wave."""
    root = VARIANT_ROOTS[variant]
    d = root / wave
    train_path = _resolve_train_path(d, variant)
    test_path = d / "test.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(
            f"Missing files for {variant}/{wave} → {train_path} / {test_path}"
        )
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    return tr, te

def wave_sort_key(wave: str):
    """Sort waves by numeric suffix if present (e.g. ..._1, ..._2, ..._10)."""
    m = re.search(r"(\d+)$", str(wave))
    return int(m.group(1)) if m else wave

# === BASE MODELS AND UTILITIES ===

def make_model(algo: str, y_tr: np.ndarray):
    """Create base binary model for given algorithm name."""
    if algo == "GBM":
        return GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE,
        )
    elif algo == "XGB" and HAS_XGB:
        pos = max(1, int((y_tr == 1).sum()))
        neg = max(1, int((y_tr == 0).sum()))
        spw = neg / pos
        return XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
            tree_method="hist",
            scale_pos_weight=spw,
        )
    elif algo == "RF":
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise RuntimeError("Unsupported algo or XGBoost not installed")

def make_sample_weights(y_tr: np.ndarray) -> np.ndarray:
    """Compute simple class weights for GBM."""
    n_pos = max(1, int((y_tr == 1).sum()))
    n_neg = max(1, int((y_tr == 0).sum()))
    w_pos = n_neg / (n_pos + n_neg)
    w_neg = n_pos / (n_pos + n_neg)
    return np.where(y_tr == 1, w_pos, w_neg)


def predict_scores(model, X: np.ndarray) -> np.ndarray:
    """Return scores for positive class."""
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[:, 1]
    else:
        try:
            dec = model.decision_function(X)
            p = (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)
        except Exception:
            p = model.predict(X).astype(float)
    return p


def threshold_for_recall(y_true: np.ndarray, scores: np.ndarray, target: float):
    """Find threshold that reaches target recall and report confusion stats."""
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    s_sorted = scores[order]
    P = int(y_true.sum())
    if P == 0:
        return 1.0, {
            "precision": 0.0,
            "recall": 0.0,
            "tp": 0,
            "fp": 0,
            "tn": int((y_true == 0).sum()),
            "fn": 0,
        }

    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)
    recall_cum = tp_cum / (P + 1e-12)

    idx = np.where(recall_cum >= target)[0]
    if len(idx) == 0:
        thr = s_sorted[-1] - 1e-12
        pred = np.ones_like(y_true)
    else:
        k = int(idx[0])
        thr = s_sorted[k]
        pred = (scores >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, P)
    return float(thr), {
        "precision": float(prec),
        "recall": float(rec),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

# === BINARY EXPERIMENT: INCREMENTAL VS FULL RETRAIN ===
def run_binary_incremental_vs_fullretrain():
    print("[BINARY] Incremental vs Full retrain experiment")
    base_root = VARIANT_ROOTS["baseline"]
    waves = list_waves(base_root) or []
    if WAVE_FILTER is not None:
        waves = [w for w in waves if WAVE_FILTER(w)]
    waves = sorted(waves, key=wave_sort_key)
    assert len(waves) >= 2, "Need at least 2 waves for incremental/full retrain experiment."
    print(f"Detected {len(waves)} wave(s): {waves}")

    thr_col = f"thr@R>={TARGET_RECALL:.2f}"

    rows = []
    out_root = PROJECT_ROOT / "6_training_mode" / "out_binary_incremental_500"
    out_root.mkdir(parents=True, exist_ok=True)

    for variant in VARIANT_ROOTS.keys():  # baseline, rose
        print(f"\n=== VARIANT: {variant} ===")

        # preload all waves
        wave_tr: Dict[str, pd.DataFrame] = {}
        wave_te: Dict[str, pd.DataFrame] = {}
        for wave in waves:
            try:
                tr_df, te_df = load_frames(variant, wave)
            except FileNotFoundError as e:
                print(f"  [skip {variant}/{wave}] {e}")
                continue
            wave_tr[wave] = tr_df
            wave_te[wave] = te_df

        usable_waves = [w for w in waves if w in wave_tr and w in wave_te]
        if len(usable_waves) < 2:
            print(f"  Not enough waves with data for variant={variant}, skip.")
            continue

        # B2…B5 → start from index 1
        for idx in range(1, len(usable_waves)):
            wave_test = usable_waves[idx]
            wave_prev = usable_waves[idx - 1]

            print(f"\n  [WAVE] test={wave_test} (prev={wave_prev})")

            for algo in ALGOS:
                if algo == "XGB" and not HAS_XGB:
                    print("    [warn] xgboost not installed, skip XGB")
                    continue

                # Incremental mode
                train_waves_inc = [wave_prev]
                tr_inc = pd.concat(
                    [wave_tr[w] for w in train_waves_inc],
                    ignore_index=True
                )

                y_tr_inc = tr_inc[LABEL_COL].astype(int).to_numpy()

                drop_cols = [LABEL_COL]
                if "Suspicion Category" in tr_inc.columns:
                    drop_cols.append("Suspicion Category")
                X_tr_inc = tr_inc.drop(columns=drop_cols).to_numpy()

                te_df = wave_te[wave_test]
                y_te = te_df[LABEL_COL].astype(int).to_numpy()

                drop_cols_te = [LABEL_COL]
                if "Suspicion Category" in te_df.columns:
                    drop_cols_te.append("Suspicion Category")
                X_te = te_df.drop(columns=drop_cols_te).to_numpy()

                model_inc = make_model(algo, y_tr_inc)
                sw_inc = make_sample_weights(y_tr_inc) if algo == "GBM" else None
                if sw_inc is not None:
                    model_inc.fit(X_tr_inc, y_tr_inc, sample_weight=sw_inc)
                else:
                    model_inc.fit(X_tr_inc, y_tr_inc)

                p_inc = predict_scores(model_inc, X_te)
                roc_inc = roc_auc_score(y_te, p_inc)
                prc_inc = average_precision_score(y_te, p_inc)
                thr_inc, at_inc = threshold_for_recall(y_te, p_inc, TARGET_RECALL)
                n_test = len(y_te)
                fp_per_1000_inc = at_inc["fp"] / max(1, n_test) * 1000.0

                print(
                    f"    [INCREMENTAL] algo={algo}, test_wave={wave_test}: "
                    f"ROC-AUC={roc_inc:.4f}, PR-AUC={prc_inc:.4f}, "
                    f"precision@R={at_inc['precision']:.3f}, recall={at_inc['recall']:.3f}, "
                    f"FP/1000={fp_per_1000_inc:.3f}"
                )

                rows.append(
                    {
                        "mode": "incremental",
                        "variant": variant,
                        "algo": algo,
                        "train_waves": ",".join(train_waves_inc),
                        "test_wave": wave_test,
                        "roc_auc": roc_inc,
                        "pr_auc": prc_inc,
                        thr_col: thr_inc,
                        "precision@R": at_inc["precision"],
                        "recall@thr": at_inc["recall"],
                        "TP": at_inc["tp"],
                        "FP": at_inc["fp"],
                        "TN": at_inc["tn"],
                        "FN": at_inc["fn"],
                        "FP_per_1000": fp_per_1000_inc,
                        "n_test": n_test,
                    }
                )

                # Full retrain mode: train on all previous waves (B1…B_{t-1})
                train_waves_full = usable_waves[:idx]
                tr_full = pd.concat(
                    [wave_tr[w] for w in train_waves_full],
                    ignore_index=True
                )

                y_tr_full = tr_full[LABEL_COL].astype(int).to_numpy()

                drop_cols_full = [LABEL_COL]
                if "Suspicion Category" in tr_full.columns:
                    drop_cols_full.append("Suspicion Category")
                X_tr_full = tr_full.drop(columns=drop_cols_full).to_numpy()

                model_full = make_model(algo, y_tr_full)
                sw_full = make_sample_weights(y_tr_full) if algo == "GBM" else None
                if sw_full is not None:
                    model_full.fit(X_tr_full, y_tr_full, sample_weight=sw_full)
                else:
                    model_full.fit(X_tr_full, y_tr_full)

                p_full = predict_scores(model_full, X_te)
                roc_full = roc_auc_score(y_te, p_full)
                prc_full = average_precision_score(y_te, p_full)
                thr_full, at_full = threshold_for_recall(y_te, p_full, TARGET_RECALL)
                fp_per_1000_full = at_full["fp"] / max(1, n_test) * 1000.0

                print(
                    f"    [FULL]        algo={algo}, test_wave={wave_test}: "
                    f"ROC-AUC={roc_full:.4f}, PR-AUC={prc_full:.4f}, "
                    f"precision@R={at_full['precision']:.3f}, recall={at_full['recall']:.3f}, "
                    f"FP/1000={fp_per_1000_full:.3f}"
                )

                rows.append(
                    {
                        "mode": "full_retrain",
                        "variant": variant,
                        "algo": algo,
                        "train_waves": ",".join(train_waves_full),
                        "test_wave": wave_test,
                        "roc_auc": roc_full,
                        "pr_auc": prc_full,
                        thr_col: thr_full,
                        "precision@R": at_full["precision"],
                        "recall@thr": at_full["recall"],
                        "TP": at_full["tp"],
                        "FP": at_full["fp"],
                        "TN": at_full["tn"],
                        "FN": at_full["fn"],
                        "FP_per_1000": fp_per_1000_full,
                        "n_test": n_test,
                    }
                )

    if not rows:
        print("No rows produced for incremental/full retrain experiment.")
        return

    res = pd.DataFrame(rows)
    res.to_csv(out_root / "binary_incremental_fullretrain_by_wave.csv", index=False)

    # summary over waves
    def agg(df):
        return pd.Series(
            {
                "waves": len(df),
                "roc_auc_mean": df["roc_auc"].mean(),
                "pr_auc_mean": df["pr_auc"].mean(),
                "precision@R_mean": df["precision@R"].mean(),
                "recall@thr_mean": df["recall@thr"].mean(),
                "FP_per_1000_mean": df["FP_per_1000"].mean(),
            }
        )

    summary = res.groupby(["mode", "variant", "algo"], as_index=False).apply(agg)
    summary.to_csv(out_root / "binary_incremental_fullretrain_summary.csv", index=False)


if __name__ == "__main__":
    run_binary_incremental_vs_fullretrain()

[BINARY] Incremental vs Full retrain experiment
Detected 5 wave(s): ['synthetic_transactions_structured_500_1', 'synthetic_transactions_structured_500_2', 'synthetic_transactions_structured_500_3', 'synthetic_transactions_structured_500_4', 'synthetic_transactions_structured_500_5']

=== VARIANT: baseline ===

  [WAVE] test=synthetic_transactions_structured_500_2 (prev=synthetic_transactions_structured_500_1)
    [INCREMENTAL] algo=GBM, test_wave=synthetic_transactions_structured_500_2: ROC-AUC=0.7520, PR-AUC=0.3400, precision@R=0.049, recall=0.981, FP/1000=877.921
    [FULL]        algo=GBM, test_wave=synthetic_transactions_structured_500_2: ROC-AUC=0.7520, PR-AUC=0.3400, precision@R=0.049, recall=0.981, FP/1000=877.921
    [INCREMENTAL] algo=XGB, test_wave=synthetic_transactions_structured_500_2: ROC-AUC=0.6830, PR-AUC=0.2780, precision@R=0.046, recall=0.981, FP/1000=934.752
    [FULL]        algo=XGB, test_wave=synthetic_transactions_structured_500_2: ROC-AUC=0.6830, PR-AUC=0.2780, 

# OOF + PLATT SCALING + META MODEL (PER-WAVE FULL RETRAIN) 500

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List
import re

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# CONFIG (BINARY ONLY)

# Binary target (0/1)
LABEL_COL = "Is_Suspicious"
# Scenario column
SCENARIO_COL = "Suspicion Category"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "4_class_balance").exists():
    parent = PROJECT_ROOT.parent
    if (parent / "4_class_balance").exists():
        PROJECT_ROOT = parent
    else:
        raise RuntimeError(f"Cannot find '4_class_balance' folder from cwd={Path.cwd()}")

# Data variants (only baseline and rose, with scenario column inside)
VARIANT_ROOTS: Dict[str, Path] = {
    "baseline": PROJECT_ROOT / "4_class_balance" / "baseline_500",
    "rose":     PROJECT_ROOT / "4_class_balance" / "rose-500",
}

VARIANT_TRAIN_FILE = {
    "baseline": "train_baseline.csv",
    "rose":     "train_rose.csv",
}

# Base binary models (GBM, XGB, RF)
ALGOS = ["GBM", "XGB", "RF"]

N_FOLDS = 5
TARGET_RECALL = 0.98
RANDOM_STATE = 42

# Optional wave filter
WAVE_FILTER = None

# HELPERS
def list_waves(root: Path) -> List[str]:
    """Return sorted list of wave directory names under root."""
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])

def _resolve_train_path(dir_path: Path, variant: str) -> Path:
    """Return train file path for variant inside dir_path."""
    primary = dir_path / VARIANT_TRAIN_FILE[variant]
    if primary.exists():
        return primary
    fallback = dir_path / "train.csv"
    if fallback.exists():
        return fallback
    return primary

def load_frames(variant: str, wave: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load train/test DataFrames for given variant and wave."""
    root = VARIANT_ROOTS[variant]
    d = root / wave
    train_path = _resolve_train_path(d, variant)
    test_path = d / "test.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(
            f"Missing files for {variant}/{wave} → {train_path} / {test_path}"
        )
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    return tr, te

def split_xy(
    tr: pd.DataFrame,
    te: pd.DataFrame,
    label_col: str = LABEL_COL,
    scenario_col: str = SCENARIO_COL,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """Split DataFrames into X/y for binary classification."""
    if label_col not in tr.columns or label_col not in te.columns:
        raise KeyError(f"Column '{label_col}' not found in train/test")

    y_tr = tr[label_col].astype(int).to_numpy()
    y_te = te[label_col].astype(int).to_numpy()

    drop_cols = [label_col]
    if scenario_col in tr.columns:
        drop_cols.append(scenario_col)

    feature_cols = [c for c in tr.columns if c not in drop_cols]
    X_tr = tr[feature_cols].to_numpy()
    X_te = te[feature_cols].to_numpy()
    return X_tr, y_tr, X_te, y_te, feature_cols

def wave_sort_key(wave: str):
    """Sort waves by numeric suffix if present (e.g. ..._1, ..._2, ..._10)."""
    m = re.search(r"(\d+)$", str(wave))
    return int(m.group(1)) if m else wave

# MODELS & UTILS (BINARY)
def make_model(algo: str, y_tr: np.ndarray):
    """Create base binary model for given algorithm name."""
    if algo == "GBM":
        return GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE,
        )
    elif algo == "XGB" and HAS_XGB:
        pos = max(1, int((y_tr == 1).sum()))
        neg = max(1, int((y_tr == 0).sum()))
        spw = neg / pos
        return XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
            tree_method="hist",
            scale_pos_weight=spw,
        )
    elif algo == "RF":
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise RuntimeError("Unsupported algo or XGBoost not installed")

def make_sample_weights(y_tr: np.ndarray) -> np.ndarray:
    """Compute simple class weights for GBM."""
    n_pos = max(1, int((y_tr == 1).sum()))
    n_neg = max(1, int((y_tr == 0).sum()))
    w_pos = n_neg / (n_pos + n_neg)
    w_neg = n_pos / (n_pos + n_neg)
    return np.where(y_tr == 1, w_pos, w_neg)

def predict_scores(model, X: np.ndarray) -> np.ndarray:
    """Return scores for positive class."""
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[:, 1]
    else:
        try:
            dec = model.decision_function(X)
            p = (dec - dec.min()) / (dec.max() - dec.min() + 1e-12)
        except Exception:
            p = model.predict(X).astype(float)
    return p

def threshold_for_recall(y_true: np.ndarray, scores: np.ndarray, target: float):
    """Find threshold that reaches target recall and report confusion stats."""
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    s_sorted = scores[order]
    P = int(y_true.sum())
    if P == 0:
        return 1.0, {
            "precision": 0.0,
            "recall": 0.0,
            "tp": 0,
            "fp": 0,
            "tn": int((y_true == 0).sum()),
            "fn": 0,
        }

    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)
    recall_cum = tp_cum / (P + 1e-12)

    idx = np.where(recall_cum >= target)[0]
    if len(idx) == 0:
        thr = s_sorted[-1] - 1e-12
        pred = np.ones_like(y_true)
    else:
        k = int(idx[0])
        thr = s_sorted[k]
        pred = (scores >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, P)
    return float(thr), {
        "precision": float(prec),
        "recall": float(rec),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

# OOF + PLATT (BINARY ONLY)
def compute_oof_and_test(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    algo: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute OOF scores and test scores for one binary model with K-fold CV."""
    n_train = len(y_tr)
    oof_scores = np.zeros(n_train, dtype=float)
    test_scores_folds: List[np.ndarray] = []

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (idx_tr, idx_val) in enumerate(skf.split(X_tr, y_tr), 1):
        model = make_model(algo, y_tr[idx_tr])
        sw = make_sample_weights(y_tr[idx_tr]) if algo == "GBM" else None

        if sw is not None:
            model.fit(X_tr[idx_tr], y_tr[idx_tr], sample_weight=sw)
        else:
            model.fit(X_tr[idx_tr], y_tr[idx_tr])

        val_scores = predict_scores(model, X_tr[idx_val])
        oof_scores[idx_val] = val_scores

        te_scores = predict_scores(model, X_te)
        test_scores_folds.append(te_scores)

    test_scores = np.mean(test_scores_folds, axis=0)
    return oof_scores, test_scores

def fit_platt(oof_scores: np.ndarray, y_tr: np.ndarray) -> LogisticRegression:
    """Fit Platt calibrator on OOF scores."""
    lr = LogisticRegression(solver="lbfgs", max_iter=1000)
    lr.fit(oof_scores.reshape(-1, 1), y_tr)
    return lr

def apply_platt(calibrator: LogisticRegression, scores: np.ndarray) -> np.ndarray:
    """Apply Platt scaling to raw scores."""
    return calibrator.predict_proba(scores.reshape(-1, 1))[:, 1]

def run_binary_oof_platt():
    print("VARIANT_ROOTS:", {k: str(v) for k, v in VARIANT_ROOTS.items()})
    waves = list_waves(VARIANT_ROOTS["baseline"]) or []
    if WAVE_FILTER is not None:
        waves = [w for w in waves if WAVE_FILTER(w)]
    waves = sorted(waves, key=wave_sort_key)
    assert waves, "No waves found in baseline. Check data paths."
    print(f"Detected {len(waves)} wave(s): {waves}")

    thr_col = f"thr@R>={TARGET_RECALL:.2f}"

    binary_rows = []

    out_root = Path("./out_oof_meta_500")
    out_root.mkdir(parents=True, exist_ok=True)

    for variant in VARIANT_ROOTS.keys():  # baseline, rose
        for wave in waves:
            try:
                tr_df, te_df = load_frames(variant, wave)
            except (FileNotFoundError, KeyError) as e:
                print(f"[skip {variant}/{wave}] {e}")
                continue

            X_tr, y_tr, X_te, y_te, feature_cols = split_xy(tr_df, te_df)

            print("\n=============================================================")
            print(f"[RUN] variant={variant}, wave={wave}")
            print("train:", X_tr.shape, "test:", X_te.shape)

            # DataFrames with OOF / test predictions for future stacking
            tr_meta = tr_df.copy()
            te_meta = te_df.copy()

            active_algos = [a for a in ALGOS if not (a == "XGB" and not HAS_XGB)]

            for algo in active_algos:
                print(f"  [BINARY] algo={algo} → OOF + Platt")
                oof_raw, test_raw = compute_oof_and_test(X_tr, y_tr, X_te, algo)
                calibrator = fit_platt(oof_raw, y_tr)
                oof_cal = apply_platt(calibrator, oof_raw)
                test_cal = apply_platt(calibrator, test_raw)

                # Binary metrics on this wave (calibrated scores)
                roc = roc_auc_score(y_te, test_cal)
                prc = average_precision_score(y_te, test_cal)
                thr, at = threshold_for_recall(y_te, test_cal, TARGET_RECALL)
                n_test = len(y_te)
                fp_per_1000 = at["fp"] / max(1, n_test) * 1000.0

                print(
                    f"    [TEST] wave={wave}, variant={variant}, algo={algo}: "
                    f"ROC-AUC={roc:.4f}, PR-AUC={prc:.4f}, "
                    f"precision@R={at['precision']:.3f}, recall={at['recall']:.3f}, "
                    f"FP/1000={fp_per_1000:.3f}"
                )

                binary_rows.append(
                    {
                        "wave": wave,
                        "variant": variant,
                        "algo": algo,
                        "roc_auc": roc,
                        "pr_auc": prc,
                        thr_col: thr,
                        "precision@R": at["precision"],
                        "recall@thr": at["recall"],
                        "TP": at["tp"],
                        "FP": at["fp"],
                        "TN": at["tn"],
                        "FN": at["fn"],
                        "FP_per_1000": fp_per_1000,
                        "n_test": n_test,
                    }
                )

                # Add calibrated binary features for future meta-model
                col_tr = f"oof_{algo}_bin"
                col_te = f"pred_{algo}_bin"
                tr_meta[col_tr] = oof_cal
                te_meta[col_te] = test_cal

            # Save extended tables with OOF / test predictions
            out_dir = out_root / variant / wave
            out_dir.mkdir(parents=True, exist_ok=True)
            tr_meta.to_csv(out_dir / "train_with_oof.csv", index=False)
            te_meta.to_csv(out_dir / "test_with_pred.csv", index=False)

    # Save per-wave binary results
    if binary_rows:
        bin_df = pd.DataFrame(binary_rows)
        bin_df.to_csv(out_root / "binary_oof_results_by_wave.csv", index=False)

        def agg(df):
            return pd.Series(
                {
                    "waves": len(df),
                    "roc_auc_mean": df["roc_auc"].mean(),
                    "pr_auc_mean": df["pr_auc"].mean(),
                    "precision@R_mean": df["precision@R"].mean(),
                    "recall@thr_mean": df["recall@thr"].mean(),
                    "FP_per_1000_mean": df["FP_per_1000"].mean(),
                }
            )

        summary = bin_df.groupby(["variant", "algo"], as_index=False).apply(agg)
        summary.to_csv(out_root / "binary_oof_summary.csv", index=False)

if __name__ == "__main__":
    run_binary_oof_platt()

VARIANT_ROOTS: {'baseline': 'c:\\Users\\T470s\\Documents\\GitHub\\Automated-Risk-Scoring-System\\4_class_balance\\baseline', 'rose': 'c:\\Users\\T470s\\Documents\\GitHub\\Automated-Risk-Scoring-System\\4_class_balance\\rose'}


AssertionError: No waves found in baseline. Check data paths.

# MULTI-CLASS MODEL (SCENARIO CLASSIFICATION) 500

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

# CONFIG
# Numeric scenario label (0 = normal, >0 = suspicious scenario)
SCENARIO_COL = "Scenario_Main"

# Folder with OOF tables from binary OOF script
OOF_ROOT = Path("./out_oof_meta_500")

VARIANTS: List[str] = ["baseline"]
RANDOM_STATE = 42

# HELPERS
def list_waves_from_oof(variant: str) -> List[str]:
    """List waves for a variant based on folders."""
    base = OOF_ROOT / variant
    if not base.exists():
        return []
    return sorted([p.name for p in base.iterdir() if p.is_dir()])

def load_oof_tables(variant: str, wave: str):
    """Load train/test OOF tables for a wave."""
    base = OOF_ROOT / variant / wave
    tr_path = base / "train_with_oof.csv"
    te_path = base / "test_with_pred.csv"
    if not tr_path.exists() or not te_path.exists():
        raise FileNotFoundError(
            f"Missing OOF files for {variant}/{wave}: {tr_path} / {te_path}"
        )
    tr = pd.read_csv(tr_path)
    te = pd.read_csv(te_path)
    return tr, te

# MULTI-CLASS META MODEL
def run_multiclass_meta():
    """Train and evaluate multi-class model on OOF scores."""
    rows = []

    for variant in VARIANTS:
        waves = list_waves_from_oof(variant)
        if not waves:
            print(f"[skip] No waves found for variant={variant}")
            continue

        print(f"\n=== VARIANT: {variant} ===")
        for wave in waves:
            try:
                tr, te = load_oof_tables(variant, wave)
            except FileNotFoundError as e:
                print(f"  [skip {variant}/{wave}] {e}")
                continue

            if SCENARIO_COL not in tr.columns or SCENARIO_COL not in te.columns:
                print(f"  [skip {variant}/{wave}] no '{SCENARIO_COL}' in tables")
                continue

            # scenario labels (numeric)
            y_tr = tr[SCENARIO_COL].fillna(0).astype(int).to_numpy()
            y_te = te[SCENARIO_COL].fillna(0).astype(int).to_numpy()

            # only suspicious transactions (scenario > 0)
            mask_tr = y_tr > 0
            mask_te = y_te > 0

            if mask_tr.sum() == 0 or mask_te.sum() == 0:
                print(f"  [skip {variant}/{wave}] no suspicious rows for multi-class")
                continue

            # OOF-based features from binary models
            oof_cols = [c for c in tr.columns if c.startswith("oof_") and c.endswith("_bin")]
            if not oof_cols:
                print(f"  [skip {variant}/{wave}] no OOF columns (oof_*_bin)")
                continue

            pred_cols = [c.replace("oof_", "pred_") for c in oof_cols]
            missing_pred = [c for c in pred_cols if c not in te.columns]
            if missing_pred:
                print(f"  [skip {variant}/{wave}] missing test preds: {missing_pred}")
                continue

            X_tr = tr.loc[mask_tr, oof_cols].to_numpy()
            X_te = te.loc[mask_te, pred_cols].to_numpy()
            y_tr_s = y_tr[mask_tr]
            y_te_s = y_te[mask_te]

            classes_train = np.unique(y_tr_s)
            if len(classes_train) < 2:
                print(f"  [skip {variant}/{wave}] only one scenario class in train, skip")
                continue

            print(
                f"  [WAVE] {wave}: "
                f"suspicious train={X_tr.shape[0]}, suspicious test={X_te.shape[0]}"
            )

            # multi-class classifier
            clf = LogisticRegression(
                multi_class="multinomial",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )
            clf.fit(X_tr, y_tr_s)
            y_pred = clf.predict(X_te)

            acc = accuracy_score(y_te_s, y_pred)
            f1_macro = f1_score(y_te_s, y_pred, average="macro")
            prec_macro = precision_score(
                y_te_s, y_pred, average="macro", zero_division=0
            )
            rec_macro = recall_score(
                y_te_s, y_pred, average="macro", zero_division=0
            )

            print(
                f"    [MULTI-CLASS TEST] acc={acc:.4f}, F1_macro={f1_macro:.4f}, "
                f"precision_macro={prec_macro:.4f}, recall_macro={rec_macro:.4f}"
            )

            rows.append(
                {
                    "variant": variant,
                    "wave": wave,
                    "n_train_suspicious": int(mask_tr.sum()),
                    "n_test_suspicious": int(mask_te.sum()),
                    "accuracy": float(acc),
                    "f1_macro": float(f1_macro),
                    "precision_macro": float(prec_macro),
                    "recall_macro": float(rec_macro),
                }
            )

            # per-class reports for this wave
            out_dir = OOF_ROOT / variant / wave
            out_dir.mkdir(parents=True, exist_ok=True)

            # confusion matrix
            classes = np.unique(y_te_s)
            cm = confusion_matrix(y_te_s, y_pred, labels=classes)
            cm_df = pd.DataFrame(cm, index=classes, columns=classes)
            cm_df.index.name = "true"
            cm_df.columns.name = "pred"
            cm_df.to_csv(out_dir / "scenario_confusion_matrix.csv")

            # per-class metrics
            rep = classification_report(
                y_te_s,
                y_pred,
                output_dict=True,
                zero_division=0,
            )
            rep_df = pd.DataFrame(rep).transpose()
            rep_df.to_csv(out_dir / "scenario_classification_report.csv")

            # save test with predictions
            te_out = te.copy()
            te_out["Scenario_Pred"] = -1
            te_out.loc[mask_te, "Scenario_Pred"] = y_pred
            te_out.to_csv(out_dir / "test_with_scenario_pred.csv", index=False)

    if not rows:
        print("No rows produced for multi-class meta experiment.")
        return

    res = pd.DataFrame(rows)
    out_path = OOF_ROOT / "multiclass_meta_summary.csv"
    res.to_csv(out_path, index=False)
    print(f"\nSaved multi-class summary to: {out_path}")

if __name__ == "__main__":
    run_multiclass_meta()